# Training loop for mini-gLM

## Load data

Import packages and configure root.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys, pathlib, subprocess
import torch
import torch.nn as nn
import torch.nn.functional as F
import pickle
!pip install pyfaidx
from pyfaidx import Fasta
from pathlib import Path
from huggingface_hub import hf_hub_download
from src.transformer import MoETransformer
from src.tokenize import BPETokenizer
from torch.utils.data import DataLoader, Dataset, BatchSampler
from torch.nn.utils.rnn import pad_sequence

# Configure root
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/mini-gLM.git"
repo_dir = Path("mini-gLM")
if COLAB:
    root = Path("/content/mini-gLM")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])
else:
    root = Path.cwd().parent
sys.path.insert(0, str(root))

# Use GPU if available
torch.manual_seed(111)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Model architecture

In [ ]:
class DenseGLM(nn.Module):
    def __init__(self, vocab_size, num_blocks, d_model, num_heads, h_dim, num_experts, top_k, p_drop):

        self.vocab_size = vocab_size
        self.num_blocks = num_blocks
        self.d_model = d_model
        self.num_heads = num_heads
        self.h_dim = h_dim
        self.num_experts = num_experts
        self.top_k = top_k
        self.p_drop = p_drop

        self.embedding = nn.Embedding(
            num_embeddings = self.vocab_size,
            embedding_dim = self.d_model
        )

        self.moe_transformer = MoETransformer(
            d_model = self.d_model,
            num_heads = self.num_heads,
            h_dim = self.h_dim,
            num_experts = self.num_experts,
            top_k = self.top_k,
            p_drop = self.p_drop
        )

        self.model = nn.ModuleList([
            self.moe_transformer for _ in range(num_blocks)
        ])

        self.final_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):

        x = self.model(x)
        logits = self.final_head(x)

        return logits

## Training

Custom Dataset and Dataloader for dynamic batching and MLM.

In [10]:
import torch

B, L = 10, 5

arr = torch.rand(B, L)
mask = torch.rand(B, L) < 0.15

arr[mask] = 10000

arr[[1, 2], [2,3]]


tensor([0.9133, 0.6214])

https://www.youtube.com/watch?v=JDy58DtZC_g

https://medium.com/@haleema.ramzan/how-to-build-a-custom-batch-sampler-in-pytorch-ce04161583ee

MLM masking.

In [ ]:
def train_glm(epochs, lr, ):

    # Load train and val data
    train_data = hg38Data()
    val_data  = hg38Data()

    train_loader = DataLoader(
        dataset = train_data, 
        batch_sampler = DynamicBatchSampler(),
        collate_fn = dynamic_collate       
    )
    val_loader = DataLoader(
        val_data,
        batch_sampler = DynamicBatchSampler(),
        collate_fn = dynamic_collate   
    )

    model = 
    for epoch in range(epochs):

        batch = 

oragnize = 

